In [1]:
import pandas as pd
from catboost import CatBoostClassifier, Pool, metrics, cv
import numpy as np


In [2]:
df = pd.read_csv('dataML_partnum2.csv')
dfx = df.drop(['Company','CreatedBy','CreatedDate','DaysOfSupply', 'LeadTime', 'Length', 'Width', 'Height', 'Gross Weight', 'Net Weight', 'ClassBuyerID', 'PlantBuyerID'], axis=1)



dfx.info()



<class 'pandas.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Part Part Num           183 non-null    str    
 1   Description             183 non-null    str    
 2   Chinese Description     183 non-null    str    
 3   Type                    183 non-null    str    
 4   Class ID                178 non-null    str    
 5   Part Ref Category       176 non-null    str    
 6   Track Lots              183 non-null    bool   
 7   IUM                     183 non-null    str    
 8   PUM                     183 non-null    str    
 9   Conversion Factor       14 non-null     float64
 10  Supplier                69 non-null     str    
 11  Planner                 93 non-null     str    
 12  NonStock                183 non-null    bool   
 13  Is PLPart               183 non-null    bool   
 14  Plant Mfg Lot Multiple  183 non-null    int64  
 15  

In [3]:
dfx['Class ID'] = dfx['Class ID'].fillna('empty')
dfx['Part Ref Category'] = dfx['Part Ref Category'].fillna('empty')
dfx['Conversion Factor'] = dfx['Conversion Factor'].fillna(0)
dfx['Supplier'] = dfx['Supplier'].fillna('empty')
dfx['Planner'] = dfx['Planner'].fillna('empty')

boolMap = {True: 1, False: 0}
dfx['Track Lots'] = dfx['Track Lots'].map(boolMap)
dfx['NonStock'] = dfx['NonStock'].map(boolMap)
dfx['Is PLPart'] = dfx['Is PLPart'].map(boolMap)
dfx['Conversion Factor'] = dfx['Conversion Factor'].astype(str)


X = dfx.drop('Correct?', axis=1)
y = dfx['Correct?']
dfx.info()

<class 'pandas.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Part Part Num           183 non-null    str  
 1   Description             183 non-null    str  
 2   Chinese Description     183 non-null    str  
 3   Type                    183 non-null    str  
 4   Class ID                183 non-null    str  
 5   Part Ref Category       183 non-null    str  
 6   Track Lots              183 non-null    int64
 7   IUM                     183 non-null    str  
 8   PUM                     183 non-null    str  
 9   Conversion Factor       183 non-null    str  
 10  Supplier                183 non-null    str  
 11  Planner                 183 non-null    str  
 12  NonStock                183 non-null    int64
 13  Is PLPart               183 non-null    int64
 14  Plant Mfg Lot Multiple  183 non-null    int64
 15  MinOrderQty             183 non-nu

In [4]:
from sklearn.model_selection import train_test_split

X_base, X_test, y_base, y_test = train_test_split(X,y,train_size=0.90)
X_train, X_validation, y_train, y_validation = train_test_split(X_base, y_base, train_size=0.75)
print(len(X_base))
print(len(X_train))
print(len(X_validation))
print(len(X_test))


164
123
41
19


In [5]:
model = CatBoostClassifier(
    custom_loss = [metrics.Accuracy()],
    random_seed=42,
    logging_level='Silent'
)

In [6]:
features = np.array([0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16])

model.fit(
    X_train, y_train,
    cat_features= features,
    eval_set = (X_validation, y_validation),
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

CatBoostClassifier(custom_loss=['Accuracy'], logging_level='Silent', random_seed=42)

In [43]:
cv_params = model.get_params()
cv_params.update({
    'loss_function': metrics.Logloss()
})
cv_data = cv(
    Pool(X,y,cat_features=features),
    cv_params,
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [47]:
print(np.max(cv_data['test-Accuracy-mean']))
print(cv_data['test-Accuracy-std'][np.argmax(cv_data['test-Accuracy-mean'])])
print(np.argmax(cv_data['test-Accuracy-mean']))

0.8961748633879781
0.0189295170226107
747


In [54]:
predictions = model.predict(X_test)
print(predictions)
print(y_test)
hits = 0
for i in range(0,len(y_test)):
    if predictions[i] == y_test.to_numpy()[i]:
        hits += 1

hitRate = (hits/len(y_test))*100
print(hitRate)


[0 1 0 1 1 1 0 1 1 1 1 1 1 0 0 1 0 1 1]
153    0
82     1
16     0
44     1
117    1
108    1
15     0
42     1
135    1
138    1
60     1
39     1
143    1
23     0
26     0
74     1
171    0
91     1
47     1
Name: Correct?, dtype: int64
100.0


In [7]:
model.save_model('model.cbm')